# DeepHPM SoH — final result analysis

This notebook presents the frozen Step 6 test results produced by three fixed seeds. It does **not** load the dataset, construct a model, or train anything. All metrics and figures come from the saved Step 7 analysis artifacts.

In [ ]:
import csv
import json
from pathlib import Path
from IPython.display import Image, Markdown, display

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
ANALYSIS = ROOT / 'artifacts' / 'deephpm' / 'analysis'
FIGURES = ANALYSIS / 'figures'
summary = json.loads((ANALYSIS / 'analysis_summary.json').read_text())
summary['seeds'], summary['records'], summary['batteries']

## Headline test metrics

The ensemble is the equal-weight mean of seed predictions aligned using `(battery_id, cycle)`. The mean ± SD values describe variability across independently initialized models; the ensemble row describes the averaged prediction.

In [ ]:
m = summary['ensemble_metrics']
across = summary['across_seed_metrics']
table = f'''
| Result | MAE | RMSE | R² | Bias |
|---|---:|---:|---:|---:|
| Three-seed ensemble | {m['mae']:.5f} | {m['rmse']:.5f} | {m['r2']:.3f} | {m['bias']:.5f} |
| Seed mean ± sample SD | {across['mae']['mean']:.5f} ± {across['mae']['sample_std']:.5f} | {across['rmse']['mean']:.5f} ± {across['rmse']['sample_std']:.5f} | {across['r2']['mean']:.3f} ± {across['r2']['sample_std']:.3f} | {across['bias']['mean']:.5f} ± {across['bias']['sample_std']:.5f} |
'''
display(Markdown(table))

The positive bias means predicted SoH is, on average, higher than measured SoH. This should be reported alongside RMSE and R² because optimistic SoH estimates matter in prognostics.

In [ ]:
display(Image(filename=str(FIGURES / '01_actual_vs_predicted_soh.png'), width=800))

## Battery degradation trajectories

The batteries below were selected reproducibly as the lowest-, median-, and highest-RMSE batteries, rather than hand-picked after viewing the curves. The shaded region is ±1 standard deviation across seeds.

In [ ]:
display(Image(filename=str(FIGURES / '02_representative_battery_trajectories.png'), width=1100))

## Error and seed stability

In [ ]:
for name in ('03_per_battery_errors', '04_error_distributions', '08_seed_stability'):
    display(Image(filename=str(FIGURES / f'{name}.png'), width=1000))

## Optimization and learned physics

The loss plot shows validation curves as solid lines and training curves as dashed lines. Dotted vertical lines mark each validation-selected checkpoint. The physics figures compare the autograd derivative with the learned dynamics and show their residual.

In [ ]:
for name in ('05_training_curves', '06_physics_residuals', '07_learned_dynamics_vs_autograd'):
    display(Image(filename=str(FIGURES / f'{name}.png'), width=1000))

## Per-battery table

In [ ]:
with (ANALYSIS / 'per_battery_metrics.csv').open() as stream:
    batteries = list(csv.DictReader(stream))
worst = sorted(batteries, key=lambda row: float(row['rmse']), reverse=True)[:10]
rows = ['| Battery | Records | MAE | RMSE | R² | Bias |', '|---:|---:|---:|---:|---:|---:|']
for row in worst:
    rows.append(f"| {row['battery_id']} | {row['records']} | {float(row['mae']):.5f} | {float(row['rmse']):.5f} | {float(row['r2']):.3f} | {float(row['bias']):.5f} |")
display(Markdown('### Ten highest-RMSE batteries\n\n' + '\n'.join(rows)))

## Presentation guidance

For the main results slide, use the measured-vs-predicted figure, representative trajectories, and the ensemble metric row. Use the training and physics figures on a methodology or validation slide. SVG and PDF versions are available for lossless placement in slides.